# graph

> CodeGraph: call-graph layer for karma, backed by graph.db. Augments semantic search with structural traversal.

In [ ]:
#| default_exp graph

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import ast, re, os
from collections import defaultdict
from litesearch import *
from fastcore.all import Path, L, patch, parallel_async, tuplify, first, fdelegates, globtastic, bind, true, dict2obj
from karma.core import arun, Karma, embedder, parse, has_init
from pyan.analyzer import CallGraphVisitor
from pyan.anutils import Scope, ExecuteInInnerScope

In [ ]:
#| export
try:
	# Bug 1 fix: auto-register missing nested lambda scopes
	def _safe_enter(self):
		o = self.analyzer
		o.name_stack.append(self.scopename)
		try: inner_ns = o.get_node_of_current_namespace().get_name()
		except Exception: inner_ns = '.'.join(o.name_stack)
		if inner_ns not in o.scopes: o.scopes[inner_ns] = Scope.from_names(self.scopename, [])
		o.scope_stack.append(o.scopes[inner_ns])
		o.context_stack.append(self.scopename)
		self.inner_ns = inner_ns
		return self
	ExecuteInInnerScope.__enter__ = _safe_enter

	# Bug 2 fix: guard None namespace in postprocessing
	_ANON = {'<listcomp>','<setcomp>','<dictcomp>','<genexpr>','<lambda>'}
	_orig_gp = CallGraphVisitor.get_parent_node
	def _safe_gp(self, n):
		if n.namespace is None: return None
		return _orig_gp(self, n)
	def _safe_collapse(self):
		for name in list(self.nodes):
			if name.partition('.')[0] not in _ANON: continue
			for n in self.nodes[name]:
				if n.namespace is None: continue
				pn = self.get_parent_node(n)
				if pn is None: continue
				for n2 in self.uses_edges.get(n, []): self.add_uses_edge(pn, n2)
				n.defined = False
	CallGraphVisitor.get_parent_node = _safe_gp
	CallGraphVisitor.collapse_inner  = _safe_collapse
except ImportError: pass

In [ ]:
#| export
class CodeGraph:
    """Call graph backed by apswutils SQLite.

    Storage: four tables in graph.db —
      graph_edges  (caller, callee, kind, confidence)
      graph_nodes  (node, flavor, file, pagerank, ...)
      co_dispatch  (group_id, node)
      file_index   (path, root, last_analyzed_at)
    """
    def __init__(self, path: str | Path = ':memory:'):
        self.db = database(path)
        ge, gn, cd, fi = self.db.t.graph_edges, self.db.t.graph_nodes, self.db.t.co_dispatch, self.db.t.file_index
        ge.create(id=int, caller=str, callee=str, kind=str, confidence=float, if_not_exists=True, pk='id')
        ge.create_index(['caller'], if_not_exists=True)
        ge.create_index(['callee'], if_not_exists=True)
        ge.create_index(['kind'], if_not_exists=True)
        gn.create(node=str, flavor=str, file=str, pagerank=float, in_degree=int,
                  out_degree=int, if_not_exists=True, pk='node')
        cd.create(group_id=int, node=str, if_not_exists=True)
        fi.create(path=str, root=str, last_analyzed_at=float, if_not_exists=True, pk='path')
        fi.create_index(['root'], if_not_exists=True)
        self.ge, self.gn, self.cd, self.fi = ge, gn, cd, fi

In [ ]:
self = CodeGraph(':memory:')
tables = {r[0] for r in self.db.execute("SELECT name FROM sqlite_master WHERE type='table'")}
assert {'graph_edges','graph_nodes','co_dispatch','file_index'} <= tables, f"Missing tables: {tables}"
print("schema ok:", tables)

schema ok: {'graph_edges', 'co_dispatch', 'file_index', 'graph_nodes'}


In [ ]:
#| export
_reg_re = re.compile(
    r'^(add|register|include|on|connect|mount|sub(scribe)?|listen|use|route|handle|hook|attach|bind|plug|get|post|'
    r'put|delete|patch|head|options|before|after|middleware)(_\w+)?$', re.IGNORECASE)
_conn = {'connect', 'setup', 'register', 'init_app', 'create_app', 'mount','configure','bootstrap','install','activate'}
_web_pkg = {'fasthtml', 'flask', 'fastapi', 'starlette', 'django', 'aiohttp','sanic','tornado','litestar','blacksheep'}

In [ ]:
#| export
def _root(n):
	while isinstance(n, ast.Attribute): n = n.value
	return n.id if isinstance(n, ast.Name) else None

def _conf(root, verb, imp):
	pkg = imp.get(root) or next((v for k,v in imp.items() if k.startswith('*')), None)
	return 0.95 if pkg in _web_pkg else 0.75 if _reg_re.match(verb or '') else 0.5

def _e(caller, callee, kind, conf): return {'caller':caller,'callee':callee,'kind':kind,'confidence':conf}

def _dec_edges(n, mod, imp):
	fn = f"{mod}.{n.name}"
	return [_e(f"{mod}.{r}", fn, 'decorator', _conf(r, inner.attr, imp))
	        for dec in n.decorator_list
	        for inner in [dec.func if isinstance(dec,ast.Call) else dec]
	        if isinstance(inner,ast.Attribute) and (r := _root(inner))]

def _conn_edges(n, mod):
	if n.name not in _conn or not n.args.args: return []
	param, fn = n.args.args[0].arg, f'{mod}.{n.name}'
	return [_e(fn, f'{mod}.{a.id}', 'connect_body', 0.9)
	        for child in ast.walk(n)
	        if isinstance(child,ast.Call) and isinstance(child.func,ast.Attribute)
	        and _root(child.func) == param
	        for a in child.args if isinstance(a,ast.Name)]

def _reg_edges(n, mod, imp, all_fns):
	if not (isinstance(n,ast.Call) and isinstance(n.func,ast.Attribute) and _reg_re.match(n.func.attr)): return []
	root = _root(n.func) or 'unknown'
	return [_e(f"{mod}.{root}", f"{mod}.{a.id}", 'registration', _conf(root, n.func.attr, imp))
	        for a in n.args if isinstance(a,ast.Name) and a.id in all_fns]

def _co_edges(n, mod, top_fns):
	if not (isinstance(n,ast.Assign) and isinstance(getattr(n,'parent',None), ast.Module)): return []
	val = n.value
	grp = ([v.id for v in val.values if isinstance(v,ast.Name) and v.id in top_fns] if isinstance(val,ast.Dict) else
	       [e.id for e in val.elts if isinstance(e,ast.Name) and e.id in top_fns]
	       if isinstance(val,(ast.List,ast.Tuple)) else [])
	return [_e(f"{mod}.{a}",f"{mod}.{b}",'co_dispatch',0.8) for i,a in enumerate(grp) for b in grp[i+1:]]

def _patch_edges(n, mod):
	'fastcore @patch: @patch def fn(self:Class) → Class gains fn as method.'
	if not any(('patch' in ast.unparse(d)) for d in n.decorator_list): return []
	if not ((args := n.args.args) and (ann:=args[0].annotation)): return []
	tgts = ([ast.unparse(ann.left), ast.unparse(ann.right)]
			   if isinstance(ann, ast.BinOp) and isinstance(ann.op, ast.BitOr)
			   else [ast.unparse(ann)])
	return [_e(f"{mod}.{t}", f"{mod}.{n.name}", 'patch', 0.9) for t in tgts]

def _delegates_edges(n, mod):
	'fastcore @delegates(target): fn wraps target\'s signature — structural dependency.'
	is_nm, is_call = lambda d: isinstance(d, ast.Name) and d.id == 'delegates', lambda d: isinstance(d, ast.Call)
	is_attr = lambda d: isinstance(d, ast.Attribute) and d.attr == 'delegates'
	is_del = lambda d: is_call(d) and (is_nm(d.func) or is_attr(d.func))
	for d in n.decorator_list:
		if not (is_del(d) and (v:=d.args[0] if d.args else first(d.keywords, lambda kw:kw.arg == 'to'))): continue
		return [_e(f"{mod}.{n.name}", f"{mod}.{ast.unparse(v)}", 'delegates', 0.85)]
	return []

def dyn_edges(src:str, module:str) -> list[dict]:
	"All dynamic dispatch edges: decorator, connect_body, registration, co_dispatch."
	tree, imp, top, all_fns = parse(src)
	if tree is None: return []
	fns = L(ast.walk(tree)).filter(lambda n: isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef)))
	return (sum(fns.map(lambda n: _dec_edges(n, module, imp)), []) +
	        sum(fns.map(lambda n: _conn_edges(n, module)), []) +
			sum(fns.map(lambda n: _patch_edges(n, module)), []) +
			sum(fns.map(lambda n: _delegates_edges(n, module)), []) +
	        sum(L(ast.walk(tree)).map(lambda n: _reg_edges(n, module, imp, all_fns)), []) +
	        sum(L(tree.body).filter(lambda n: isinstance(n,ast.Assign))
	            .map(lambda n: _co_edges(n, module, top)), []))

def static_edges(sources:dict[str,str]=None, filenames:list[str]=None, root:str=None) -> tuple[list[dict], dict]:
	"Static call edges via pyan3. Provide either filenames+root or sources dict."
	assert bool(filenames) ^ bool(sources), 'Provide one of sources or filenames, but not both'
	if filenames and root is None: root = str(Path(filenames[0]).parent.parent)
	try:
		from pyan.analyzer import CallGraphVisitor
		if filenames: v=CallGraphVisitor(filenames, root=root)
		else: v= CallGraphVisitor.from_sources(tuplify((s,m) for m,s in sources.items()))
		v.postprocess()
		return ([_e(f'{c.namespace}.{c.name}', f'{t.namespace}.{t.name}', 'static', 1.0)
		         for c,ts in v.uses_edges.items() if c.namespace and c.namespace != '*'
		         for t in ts if '^^^' not in (t.name or '')],
		        {f"{n.namespace}.{n.name}":
			         {'node':f"{n.namespace}.{n.name}",
			          'flavor':str(n.flavor).split('.')[-1].lower(), 'file':n.filename or ''}
		         for nlist in v.nodes.values() for n in nlist if hasattr(n,'name') and n.namespace})
	except Exception as ex: print(f"pyan3 static analysis failed: {ex}"); return [], {}

In [ ]:
_SRC = '''
import flask
app = flask.Flask(__name__)

@app.route('/hello')
def hello(): pass

def setup(other_app):
    other_app.register(hello)
'''
edges = dyn_edges(_SRC, 'mymod')
kinds = {e['kind'] for e in edges}
assert 'decorator' in kinds, f"Expected decorator edge, got: {edges}"
assert 'registration' in kinds, f"Expected registration edge, got: {edges}"
assert dyn_edges('def (broken', 'bad') == L()
print("_dyn_edges ok:", edges)

_dyn_edges ok: [{'caller': 'mymod.app', 'callee': 'mymod.hello', 'kind': 'decorator', 'confidence': 0.75}, {'caller': 'mymod.setup', 'callee': 'mymod.hello', 'kind': 'connect_body', 'confidence': 0.9}, {'caller': 'mymod.other_app', 'callee': 'mymod.hello', 'kind': 'registration', 'confidence': 0.75}]


In [ ]:
_DELEGATES_SRC = '''
from fastcore.all import delegates

def target(x, y, z=1): pass

@delegates(target)
def wrapper_pos(**kw): pass

@delegates(to=target)
def wrapper_kw(**kw): pass

@delegates
def bare(**kw): pass

@delegates()
def empty(**kw): pass
'''
dyn_edges(_DELEGATES_SRC, 'mymod')

[{'caller': 'mymod.wrapper_pos',
  'callee': 'mymod.target',
  'kind': 'delegates',
  'confidence': 0.85},
 {'caller': 'mymod.wrapper_kw',
  'callee': 'mymod.to=target',
  'kind': 'delegates',
  'confidence': 0.85}]

In [ ]:
_PATCH_SRC = '''
from fastcore.all import patch, delegates
from apswutils.db import Database, Table, View

@patch
def basic(self:Database, x): pass

@patch(as_prop=True)
def prop(self:Database): pass

@patch(cls_method=True)
def cls_m(cls:Database): pass

@patch
def no_ann(self): pass

@patch
def union_m(self:Table|View): pass
'''
dyn_edges(_PATCH_SRC, 'mymod')

[{'caller': 'mymod.Database',
  'callee': 'mymod.basic',
  'kind': 'patch',
  'confidence': 0.9},
 {'caller': 'mymod.Database',
  'callee': 'mymod.prop',
  'kind': 'patch',
  'confidence': 0.9},
 {'caller': 'mymod.Database',
  'callee': 'mymod.cls_m',
  'kind': 'patch',
  'confidence': 0.9},
 {'caller': 'mymod.Table',
  'callee': 'mymod.union_m',
  'kind': 'patch',
  'confidence': 0.9},
 {'caller': 'mymod.View',
  'callee': 'mymod.union_m',
  'kind': 'patch',
  'confidence': 0.9}]

In [ ]:
edges, meta = static_edges({"mod": "def foo(): pass\ndef bar(): foo()"})
meta

{'mod.foo': {'node': 'mod.foo', 'flavor': 'function', 'file': 'mod'},
 'mod.foo.^^^argument^^^': {'node': 'mod.foo.^^^argument^^^',
  'flavor': 'unspecified',
  'file': 'mod'},
 'mod.bar.^^^argument^^^': {'node': 'mod.bar.^^^argument^^^',
  'flavor': 'unspecified',
  'file': 'mod'},
 'mod.bar': {'node': 'mod.bar', 'flavor': 'function', 'file': 'mod'}}

In [ ]:
assert all(k in edges[0] for k in ('caller','callee','kind','confidence'))
assert all(e['kind'] == 'static' for e in edges)

In [ ]:
static_edges({"bad": "def (: !!!"})

pyan3 static analysis failed: invalid syntax (bad, line 1)


([], {})

In [ ]:
#| export
def _nodes(db):
    'All node connectors buy caller and callee columns. O(E).'
    return L(r['node'] for r in db.q('''SELECT DISTINCT caller node FROM graph_edges
                                     UNION SELECT DISTINCT callee FROM graph_edges'''))

def _pagerank(db, nodes:set=None, alpha=0.85, iters=50, tol=1e-6):
    'Iterative PageRank with precomputed adjacency — O(E + N·iters).'
    nodes = nodes or _nodes(db)
    if not nodes: return {}
    n = len(nodes)
    pr = {nd: 1/n for nd in nodes}
    out,in_e = defaultdict(int), defaultdict(list)
    for r in db.t.graph_edges(select='caller,callee'):
	    out[r['caller']] += 1
	    in_e[r['callee']].append(r['caller'])
    for _ in range(iters):
	    new = {nd:(1-alpha)/n + alpha*sum(pr.get(c, 0)/(out[c] or 1) for c in in_e[nd]) for nd in nodes}
	    if max(abs(new[nd]-pr[nd]) for nd in nodes) < tol: break
	    pr = new
    return pr

In [ ]:
#| export
@patch
def _centrality(self: CodeGraph, nodes:set=None):
	db = self.db
	pr = _pagerank(db, nodes)
	in_d  = {r['nd']:r['n'] for r in db.q('SELECT callee nd,COUNT(*) n FROM graph_edges GROUP BY callee')}
	out_d = {r['nd']:r['n'] for r in db.q('SELECT caller nd,COUNT(*) n FROM graph_edges GROUP BY caller')}
	db.t.graph_nodes.insert_all([{'node':nd, 'pagerank':round(pr.get(nd,0),5), 'in_degree':in_d.get(nd,0),
	                              'out_degree':out_d.get(nd,0)} for nd in set(pr)], upsert=True, pk='node')
	return self

@patch
def _add_dyn(self:CodeGraph, mod_map:dict):
	"Add dynamic edges; mod_map values are src strings or Path objects."
	existing = {(r['caller'],r['callee']) for r in self.db.t.graph_edges(select='caller,callee')}
	d_edges, groups, new_nodes = [], [], set()
	for mod, val in mod_map.items():
		src = Path(val).read_text(errors='replace') if isinstance(val, os.PathLike) else val
		for e in dyn_edges(src, mod):
			fr,t = e['caller'], e['callee']
			if e['kind'] == 'co_dispatch':
				for grp in groups:
					if {fr,t} & grp: grp |= {fr,t}; break
				else: groups.append({fr,t})
			elif (fr,t) not in existing:
				d_edges.append(e)
				existing.add((fr,t))
			new_nodes |= {fr,t}
	if d_edges: self.db.t.graph_edges.insert_all(d_edges)
	if groups: self.db.t.co_dispatch.insert_all([{'group_id':i,'node':nd} for i,grp in enumerate(groups) for nd in grp])
	return new_nodes

def _lambda_nodes(sources:dict[str,str]=None, filenames:list[str]=None, root:str=None) -> dict:
	'Extract module-level `name = lambda ...` nodes; same signature as static_edges.'
	if filenames:
		fn = lambda p: '.'.join(Path(p).relative_to(root).with_suffix('').parts)
		sources = {fn(p): (Path(p).read_text(errors='replace'), str(p)) for p in filenames}
	else: sources = {mod: (src, '') for mod, src in (sources or {}).items()}
	return {f"{mod}.{t.id}": {'node':f"{mod}.{t.id}", 'flavor':'function', 'file':file}
			for mod, (src, file) in sources.items()
			for tree, *_ in [parse(src)] if tree
			for n in tree.body
			if isinstance(n, ast.Assign) and isinstance(n.value, ast.Lambda)
			for t in n.targets if isinstance(t, ast.Name)}

@patch
def _add_static(self:CodeGraph, sources:dict[str,str]=None, filenames:list[str]=None, root:str=None):
	"Add static edges from sources dict. Uses pyan3 for full-corpus analysis."
	s_edges, meta = static_edges(sources=sources, filenames=filenames, root=root)
	meta |= _lambda_nodes(sources=sources, filenames=filenames, root=root)
	if s_edges: self.db.t.graph_edges.insert_all(s_edges)
	if meta: self.db.t.graph_nodes.insert_all(meta.values(), upsert=True, pk='node')
	return set(meta) | set(L(s_edges).attrgot('caller')) | set(L(s_edges).attrgot('callee'))

@patch
def from_sources(self:CodeGraph, sources:dict[str,str]):
	'Build from {module_name: source_str}.'
	return self._centrality(self._add_static(sources=sources) | self._add_dyn(sources))

In [ ]:
g_pr = CodeGraph(':memory:')
g_pr.db['graph_edges'].insert_all([
    {'caller':'A','callee':'B','kind':'static','confidence':1.0},
    {'caller':'B','callee':'C','kind':'static','confidence':1.0},
    {'caller':'A','callee':'C','kind':'static','confidence':1.0},
])
pr = _pagerank(g_pr.db)
print(pr)
assert 'C' in pr, "C should be in pagerank"
assert pr['C'] > pr['A'], f"C (most linked-to) should rank higher than A: {pr}"
print("pagerank ok:", pr)

{'A': 0.05000000000000001, 'B': 0.07125000000000001, 'C': 0.13181250000000003}
pagerank ok: {'A': 0.05000000000000001, 'B': 0.07125000000000001, 'C': 0.13181250000000003}


In [ ]:
CodeGraph(':memory:').from_sources({'mod': 'def foo(): pass\ndef bar(): foo()\n'}).db.t.graph_nodes()

[{'node': 'mod.foo',
  'flavor': 'function',
  'file': 'mod',
  'pagerank': 0.06938,
  'in_degree': 1,
  'out_degree': 0},
 {'node': 'mod.foo.^^^argument^^^',
  'flavor': 'unspecified',
  'file': 'mod',
  'pagerank': 0.0375,
  'in_degree': 0,
  'out_degree': 0},
 {'node': 'mod.bar.^^^argument^^^',
  'flavor': 'unspecified',
  'file': 'mod',
  'pagerank': 0.0375,
  'in_degree': 0,
  'out_degree': 0},
 {'node': 'mod.bar',
  'flavor': 'function',
  'file': 'mod',
  'pagerank': 0.0375,
  'in_degree': 0,
  'out_degree': 1}]

In [ ]:
#| export
@patch
def process_files(self:CodeGraph, files, root=None):
	if root is None or not files: root = files[0] if files else '.'
	pkg_dir = Path(str(files[0])).parent
	while has_init(pkg_dir): pkg_dir = pkg_dir.parent
	root = str(pkg_dir)
	nodes = self._add_static(filenames=files, root=root)
	fn = lambda p: '.'.join(Path(p).relative_to(root).with_suffix('').parts)
	nodes |= self._add_dyn({fn(p): Path(p) for p in files})
	return self._centrality(nodes)

@patch
@fdelegates(globtastic)
def from_dir(self:CodeGraph, dir: str | Path, **kwargs) -> 'CodeGraph':
	'Build from .py files under path — pyan reads files directly.'
	files = dir2files(dir, exts=['.py'], func=os.path.join, **kwargs)
	return self.process_files(files, Path(dir).parent)

@patch
def from_file(self: CodeGraph, path:str|Path) -> 'CodeGraph':
	'Build from a single .py file. Module name derived from filename.'
	p = Path(path).resolve()
	pkg_root, parts = p.parent, [p.with_suffix('').name]
	while has_init(pkg_root):
		parts.insert(0, pkg_root.name)
		pkg_root = pkg_root.parent
	return self._centrality(self._add_static(filenames=[str(p)], root=pkg_root) | self._add_dyn({'.'.join(parts): p}))

@patch
@fdelegates(globtastic)
def from_pkg(self: CodeGraph, pkg: str, **kwargs) -> 'CodeGraph':
    'Build from an installed package. Uses pyan3 on the package source files.'
    s = spec(pkg)
    if not s: raise ValueError(f"{pkg!r} not installed")
    return self.from_dir(Path(s.origin).parent, **kwargs)

In [ ]:
cg=CodeGraph(':memory:').from_pkg('httpx').from_pkg('fastcore').from_pkg('litesearch').from_pkg('fastlite').from_pkg('apswutils')

In [ ]:
cg.db.t.graph_nodes(select='*', where='node="httpx._api.request"',limit=3)

[{'node': 'httpx._api.request',
  'flavor': 'function',
  'file': '/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/httpx/_api.py',
  'pagerank': 8e-05,
  'in_degree': 8,
  'out_degree': 22}]

In [ ]:
cg.from_dir('../karma')

<__main__.CodeGraph>

In [ ]:
cg.db.t.graph_nodes.count

11145

In [ ]:
cg.gn('node like "%emb_doc%"')

[{'node': 'karma.core.emb_doc',
  'flavor': 'function',
  'file': '../karma/core.py',
  'pagerank': 0.00013,
  'in_degree': 0,
  'out_degree': 0}]

In [ ]:
#| export
@patch
def callers(self: CodeGraph, n: str, with_kind=False) -> L:
	fn = lambda r: (r['caller'], r['kind']) if with_kind else r['caller']
	return L(self.db.t.graph_edges(select='caller,kind', where='callee=?', where_args=[n])).map(fn)

@patch
def callees(self: CodeGraph, n: str, with_kind=False) -> L:
	fn = lambda r: (r['callee'], r['kind']) if with_kind else r['callee']
	return L(self.db.t.graph_edges(select='callee,kind', where='caller=?', where_args=[n])).map(fn)

@patch
def neighbors(self: CodeGraph, n: str, depth: int = 1) -> L:
	'callers + callees up to depth hops.'
	seen, frontier = set(), {n}
	for _ in range(depth):
		nxt = set()
		for node in frontier: nxt |= set(self.callers(node)) | set(self.callees(node))
		frontier = nxt - seen - {n}
		seen |= frontier
	return L(seen)

@patch
def co_dispatched(self: CodeGraph, node: str) -> L:
	if not (rows := self.db.t.co_dispatch(select='group_id', where='node=?', where_args=[node])): return L()
	o=self.db.t.co_dispatch(select='node', where='group_id=? and node!=?', where_args=[rows[0]['group_id'], node])
	return L(o).attrgot('node')

@patch
def ranked(self: CodeGraph, k: int = 10, module: str = None) -> L:
	'Top-k nodes by pagerank, optionally filtered to a module prefix.'
	kw = dict(where='node LIKE ?', where_args=[f"{module}.%"]) if module else {}
	return L(self.db.t.graph_nodes(select='node,pagerank', order_by='pagerank DESC', limit=k, **kw))

@patch
def short_path(self: CodeGraph, src: str, tgt: str) -> L:
	'Shortest path from src to tgt via recursive CTE. Returns L of node names.'
	sql = '''WITH RECURSIVE p(node, route, depth) AS (
              SELECT ?, ?, 0
              UNION ALL
              SELECT e.callee, p.route||','||e.callee, p.depth+1
              FROM graph_edges e JOIN p ON e.caller=p.node
              WHERE p.depth<15 AND p.route NOT LIKE '%'||e.callee||'%'
          ) SELECT route FROM p WHERE node=? LIMIT 1'''
	rows = list(self.db.q(sql, [src, src, tgt]))
	return L(rows[0]['route'].split(',')) if rows else L()

@patch
def short_paths(self: CodeGraph, nodes, k: int = 10) -> L:
	'Shortest paths between all pairs in top-k nodes via combinations.'
	return L(nodes[:k]).combinations(2).map(lambda ab: self.short_path(*ab)).filter(true).sorted(key=len, reverse=True)

@patch
def edge_kinds(self: CodeGraph) -> dict:
	return {r['kind']: r['n'] for r in self.db.q("SELECT kind, COUNT(*) n FROM graph_edges GROUP BY kind")}

@patch
def node_info(self: CodeGraph, node: str, with_kind=False) -> dict:
	meta = first(self.gn(xtra=dict(node=node))) or {}
	edges = dict(callers=self.callers(node,with_kind), callees=self.callees(node,with_kind))
	return meta | edges | dict(co_dispatched=self.co_dispatched(node))

In [ ]:
cg.db.t.graph_edges(where='caller like "%apswutils.db.Table.insert_all%" and callee like "%insert_chunk%"')

[{'id': 15264,
  'caller': 'apswutils.db.Table.insert_all',
  'callee': 'apswutils.db.Table.insert_chunk',
  'kind': 'static',
  'confidence': 1.0}]

In [ ]:
L(cg.gn(where='node like "%fastcore.%"')).attrgot('node')

(#4116) ['fastcore.str', 'fastcore._modidx.str', 'fastcore.ansi.str', 'fastcore.ansi.strip_terminal_queries.str', 'fastcore.ansi.strip_ansi.str', 'fastcore.ansi.ansi2html.str', 'fastcore.ansi.ansi2latex.str', 'fastcore.ansi._htmlconverter.str', 'fastcore.ansi._latexconverter.str', 'fastcore.ansi._ansi2anything.str', 'fastcore.basics.str', 'fastcore.basics.ifnone.str', 'fastcore.basics.maybe_attr.str', 'fastcore.basics.basic_repr.str', 'fastcore.basics.basic_repr._f.str', 'fastcore.basics.basic_repr._f.listcomp.0.str', 'fastcore.basics.basic_repr._f.genexpr.0.str', 'fastcore.basics.BasicRepr.str', 'fastcore.basics.is_array.str', 'fastcore.basics.listify.str', 'fastcore.basics.tuplify.str', 'fastcore.basics.true.str', 'fastcore.basics.NullType.str', 'fastcore.basics.tonull.str', 'fastcore.basics.get_class.str', 'fastcore.basics.mk_class.str', 'fastcore.basics.wrap_class.str', 'fastcore.basics.ignore_exceptions.str', 'fastcore.basics.exec_local.str', 'fastcore.basics.risinstance.str', 'fa

In [ ]:
cg.node_info('fastcore.foundations.L')

{'callers': [], 'callees': [], 'co_dispatched': []}

In [ ]:
n = cg.db.t.graph_nodes(where='node like "%mk_write"')[0]['node']
cg.node_info(n)

{'node': 'fastcore.xtras.mk_write',
 'flavor': 'function',
 'file': '/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/fastcore/xtras.py',
 'pagerank': 4e-05,
 'in_degree': 2,
 'out_degree': 7,
 'callers': ['fastcore.xtras.write_json', 'fastcore.xtras.Path'],
 'callees': ['fastcore.basics.patch', 'None.Path', 'None.os', 'None.parent', 'None.write_text', 'None.NoneType', 'None.int'],
 'co_dispatched': []}

In [ ]:
cg.short_path('apswutils.db.Table.upsert', 'apswutils.db.Table.insert_chunk')

['apswutils.db.Table.upsert', 'apswutils.db.Table.upsert_all', 'apswutils.db.Table.insert_all', 'apswutils.db.Table.insert_chunk']

In [ ]:
cg.ranked(5, module='litesearch')

[{'node': 'litesearch.data.kw', 'pagerank': 0.00066}, {'node': 'litesearch.utils.FastEncode._load_tok.listcomp.0', 'pagerank': 0.00035}, {'node': 'litesearch.utils.encode_pdf_images.listcomp.1', 'pagerank': 0.00035}, {'node': 'litesearch.core.vec_search', 'pagerank': 0.00034}, {'node': 'litesearch.data.file_parse.meta_', 'pagerank': 0.00034}]

In [ ]:
cg.callers('litesearch.data.kw')

['litesearch.core.get_store', 'litesearch.core.database', 'litesearch.data.pkg2chunks', 'litesearch.data.pre', 'litesearch.utils.FastEncode.encode', 'litesearch.utils.FastEncode.encode.lambda.0', 'litesearch.utils.FastEncode.encode_document', 'litesearch.utils.FastEncode.encode_query', 'litesearch.utils.FastEncodeImage.embed', 'litesearch.utils.FastEncodeImage.embed.lambda.0', 'litesearch.utils.FastEncodeMultimodal.encode_text', 'litesearch.utils.FastEncodeMultimodal.encode_image', 'litesearch.utils.encode_pdf_texts']

In [ ]:
#| export
@patch
def _drop_file(self: CodeGraph, path: str):
    'Delete all graph_edges and graph_nodes belonging to this file path.'
    if not (nodes:=self.file2nodes(path)): return
    pl, nl = ','.join('?' * len(nodes)), list(nodes)
    self.db.q(f"DELETE FROM graph_edges WHERE caller IN ({pl}) OR callee IN ({pl})",nl*2)
    self.db.q(f"DELETE FROM graph_nodes WHERE node IN ({pl})", nl)
    self.db.q(f"DELETE FROM co_dispatch WHERE node IN ({pl})", nl)
    self.db.q("DELETE FROM file_index WHERE path=?", [path])

@patch
def file2nodes(self: CodeGraph, path: str) -> set:
    'Return set of node names whose file column matches path.'
    return set(L(self.db.t.graph_nodes(select='distinct node', where='file=?', where_args=[path])).attrgot('node'))

In [ ]:
g_drop = CodeGraph(':memory:')
g_drop.db.t.graph_nodes.insert_all([
    {'node':'mod.foo','flavor':'function','file':'/proj/mod.py'},
    {'node':'mod.bar','flavor':'function','file':'/proj/mod.py'},
])
g_drop.db.t.graph_edges.insert({'caller':'mod.foo','callee':'mod.bar','kind':'static','confidence':1.0})
assert g_drop.file2nodes('/proj/mod.py') == {'mod.foo','mod.bar'}
g_drop._drop_file('/proj/mod.py')
assert g_drop.file2nodes('/proj/mod.py') == set()
assert not list(g_drop.db.execute("SELECT * FROM graph_edges"))
print("_drop_file + _nodes_for_file ok")

_drop_file + _nodes_for_file ok


In [ ]:
#| export
@patch
def _index_files(self: CodeGraph, files, sc='repo'):
	'Add or update file_index entries for these files. root is for grouping, e.g. by dir or pkg.'
	o = [dict(path=str(f), root=sc, last_analyzed_at=Path(f).stat().st_mtime) for f in files]
	self.db.t.file_index.insert_all(o, replace=True)

@patch
def _stale(self: CodeGraph, files, sc=None) -> list:
	kw = dict(where='root=?', where_args=[sc]) if sc else {}
	known = {r['path']: r['last_analyzed_at'] for r in self.db.t.file_index(**kw)}
	return [f for f in files if Path(f).exists() and known.get(str(f)) != Path(f).stat().st_mtime]

In [ ]:
#| export
@patch
def sync_dir(self: CodeGraph, dir: str | Path) -> 'CodeGraph':
	'Incremental sync for a directory: drop missing files, update changed files, add new files.'
	if not dir: print('no action. dir empty'); return self
	files = dir2files(dir, exts=['.py'], func=os.path.join)
	known=set(L(self.db.t.file_index(select='distinct path', where='root=?', where_args=[str(dir)])).attrgot('path'))
	for f in known - set(files): self._drop_file(f)
	if fs:=self._stale(files, str(dir)): self.process_files(fs, str(dir)) and self._index_files(fs, str(dir))
	return self

@patch
def sync_pkgs(self: CodeGraph, pkgs: list[str]) -> 'CodeGraph':
	'Incremental sync for packages: drop missing files, update changed files, add new files.'
	if not pkgs: print('no action. pkgs empty'); return self
	def _do(p):
		files, fi = pkg2files(p, exts=['.py'], func=os.path.join), self.db.t.file_index
		known =set(L(fi(select='distinct path', where='root=?', where_args=[p])).attrgot('path'))
		for f in known - set(files): self._drop_file(f)
		root = Path(spec(p).origin).parent.parent
		if fs:=self._stale(files, p): self.process_files(fs, root) and self._index_files(fs, p)
	arun(parallel_async(_do, pkgs))
	return self

@patch
def sync(self: CodeGraph, dir=None, pkgs=None) -> 'CodeGraph':
	'Incremental sync: from_dir per dir, from_pkg per pkg; manage file_index.'
	self.sync_dir(dir) and self.sync_pkgs(pkgs)
	return self

In [ ]:
import tempfile, time

with tempfile.TemporaryDirectory() as tmp:
	pa, pb = Path(f'{tmp}/a.py'), Path(f'{tmp}/b.py')
	pa.write_text('def foo(): pass\n'); pb.write_text('def bar(): pass\n')

	g = CodeGraph(':memory:')
	g.sync(dir=tmp)
	fi = {r['path']: r['last_analyzed_at'] for r in g.db.t.file_index.rows}
	assert str(pa) in fi and str(pb) in fi, fi

	# Modify b.py — mtime changes
	time.sleep(0.01)
	pb.write_text('def bar(): return 42\n')
	pb.touch()  # ensure mtime update
	g.sync(dir=tmp)
	fi2 = {r['path']: r['last_analyzed_at'] for r in g.db.t.file_index.rows}
	assert fi2[str(pa)] == fi[str(pa)], f"a.py should not be reprocessed: {fi2}"
	assert fi2[str(pb)] != fi[str(pb)], f"b.py should be reprocessed: {fi2}"

	# Delete a.py — should be removed from file_index
	pa.unlink()
	g.sync(dir=tmp)
	fi3 = {r['path'] for r in g.db.t.file_index.rows}
	assert str(pa) not in fi3, f"a.py should be removed: {fi3}"
	print("sync incremental ok, remaining files:", fi3)

	# add packages to codegraph
	g.sync(pkgs=['httpx', 'fastcore'])
	fi4 = {r['path'] for r in g.db.t.file_index.rows}
	assert any('httpx' in p for p in fi4), f"httpx files should be indexed: {fi4}"
	assert any('fastcore' in p for p in fi4), f"fastcore files should be indexed: {fi4}"
	print("sync_pkgs ok, indexed packages:", fi4)


no action. pkgs empty
no action. pkgs empty
no action. pkgs empty
sync incremental ok, remaining files: {'/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpiedo2k23/b.py'}
no action. dir empty
sync_pkgs ok, indexed packages: {'/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/httpx/_main.py', '/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/fastcore/meta.py', '/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/httpx/_transports/mock.py', '/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/fastcore/foundation.py', '/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/httpx/__init__.py', '/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/fastcore/__init__.py', '/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/fastcore/nb_imports.py', '/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/httpx/_decoders.py', '/

In [ ]:
#| export
@patch(as_prop=True)
def graph(self: Karma) -> CodeGraph:
	'code graph for this karma instance'
	return CodeGraph(self.root.joinpath('.karma','graph.db'))

@patch(as_prop=True)
def graphdb(self: Karma):
	'Underlying graph.db database for direct queries.'
	return self.graph.db

@patch(as_prop=True)
def gn(self:Karma):
	'graph_nodes table for direct queries.'
	return self.graph.db.t.graph_nodes

@patch(as_prop=True)
def ge(self:Karma):
	'graph_edges table for direct queries.'
	return self.graph.db.t.graph_edges

@patch(as_prop=True)
def ni(self:Karma):
	'Node info helper: graph_nodes row + callers, callees, co-dispatched nodes.'
	return self.graph.node_info

@patch(as_prop=True)
def neighbors(self:Karma):
	'Neighbors helper: callers + callees up to depth hops.'
	return self.graph.neighbors

@patch(as_prop=True)
def short_path(self:Karma):
	'Shortest path helper: shortest path between two nodes.'
	return self.graph.short_path

@patch(as_prop=True)
def short_paths(self:Karma):
	'Shortest paths helper: shortest paths between all pairs in top-k nodes.'
	return self.graph.short_paths

In [ ]:
#| export
@patch
def sync(self: Karma, pkgs=None, dir=None, emb=embedder):
	'Sync code store, env store, and code graph. Runs in a daemon thread by default.'
	dir = dir or self.root
	ts=[bind(self.update_repo, dir=dir, efn=emb), bind(self.update_pkgs, pkgs=pkgs, efn=emb),
	    bind(self.graph.sync, dir=dir, pkgs=pkgs)]
	return arun(parallel_async(lambda f: f(), ts))

@patch
def context(self: Karma,
			q: str,               # query with optional key:value filters
			limit: int = 50,
			repo: bool = True,
			env: bool = True,
			graph: bool = True,
            columns:str='content,metadata',
            sys_wide=True,
			**kw                  # forwarded to env_context / repo_context
) -> L:
	'Fan-out semantic search: parse filters, run repo + env searches, merge with chained RRF.'
	def _tag(res, pref): return L(res).map(lambda r: r | dict(_src_id=f'{pref}:{r.get('rowid', id(r))}'))
	items = []
	if repo: items.append(('repo', bind(self.repo_context, q, limit=limit*2, columns=columns, **kw)))
	if env: items.append(('env', bind(self.env_context, q, limit=limit*2, columns=columns,sys_wide=sys_wide, **kw)))
	results = L(arun(parallel_async(lambda f: (f[0],f[1]()), items)))
	results = results.map(lambda r: _tag(r[1], r[0]))
	rrf = bind(rrf_merge, limit=limit*2, id_key='_src_id')
	res = L(L(results[1:]).reduce(lambda m,rs: rrf(m,rs), results[0]))
	if not graph: return res[:limit]
	return dict2obj(res.map(lambda r : r | self.ni(r['metadata']['mod_name'])))[:limit]

In [ ]:
k = Karma(xdg_dir=Path('.'))

In [ ]:
k.sync(pkgs=['httpx', 'fastcore', 'litesearch','fastlite','apswutils','python-fasthtml','apsw','ghapi'])

[None,
 [None, None, None, None, None, None, None],
 <__main__.CodeGraph>]

In [ ]:
k.gn(where='node like "fastcore.basics.me%"', limit=1)

[{'node': 'fastcore.basics.merge.str',
  'flavor': 'attribute',
  'file': '/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/fastcore/basics.py',
  'pagerank': 3e-05,
  'in_degree': 0,
  'out_degree': 0}]

In [ ]:
k.graph.ranked(module='fastcore.basics')

[{'node': 'fastcore.basics.patch', 'pagerank': 0.00089}, {'node': 'fastcore.basics.NS.__iter__', 'pagerank': 0.00056}, {'node': 'fastcore.basics.NotStr.__iter__', 'pagerank': 0.00056}, {'node': 'fastcore.basics.GetAttr', 'pagerank': 0.00035}, {'node': 'fastcore.basics.NotStr', 'pagerank': 0.0003}, {'node': 'fastcore.basics.listify', 'pagerank': 0.00025}, {'node': 'fastcore.basics.NS', 'pagerank': 0.0002}, {'node': 'fastcore.basics.adict', 'pagerank': 0.00018}, {'node': 'fastcore.basics.AttrDict', 'pagerank': 0.00017}, {'node': 'fastcore.basics.not_._f', 'pagerank': 0.00014}]

In [ ]:
k.ni('fastcore.basics.merge')

{'node': 'fastcore.basics.merge',
 'flavor': 'function',
 'file': '/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/fastcore/basics.py',
 'pagerank': 3e-05,
 'in_degree': 1,
 'out_degree': 12,
 'callers': ['fastcore.script.call_parse._f'],
 'callees': ['fastcore.docscrape.NumpyDocString.__iter__', 'fastcore.nbio.Notebook.__iter__', 'fastcore.xtras.IterLen.__iter__', 'fastcore.basics.NotStr.__iter__', 'None.__next__', 'fastcore.xml.FT.__iter__', 'None.ds', 'fastcore.basics.NS.__iter__', 'fastcore.foundation.CollBase.__iter__', 'fastcore.xtras._save_iter.__iter__', 'fastcore.xtras.CachedIter.__iter__', 'fastcore.xtras.SaveReturn.__iter__'],
 'co_dispatched': []}

In [ ]:
first(k.context('how do I merge dicts package:fastcore path:basics', limit=1))

```python
{ '_rrf_score': 0.016666666666666666,
  '_src_id': 'env:641',
  'callees': ['fastcore.docscrape.NumpyDocString.__iter__', 'fastcore.nbio.Notebook.__iter__', 'fastcore.xtras.IterLen.__iter__', 'fastcore.basics.NotStr.__iter__', 'None.__next__', 'fastcore.xml.FT.__iter__', 'None.ds', 'fastcore.basics.NS.__iter__', 'fastcore.foundation.CollBase.__iter__', 'fastcore.xtras._save_iter.__iter__', 'fastcore.xtras.CachedIter.__iter__', 'fastcore.xtras.SaveReturn.__iter__'],
  'callers': ['fastcore.script.call_parse._f'],
  'co_dispatched': [],
  'content': 'def merge(*ds):\n'
             '    "Merge all dictionaries in `ds`"\n'
             '    return {k:v for d in ds if d is not None for k,v in '
             'd.items()}',
  'file': '/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/fastcore/basics.py',
  'flavor': 'function',
  'in_degree': 1,
  'metadata': { 'end_lineno': 657,
                'lang': '.py',
                'lineno': 655,
                'mod_name': 'fastcore.basics.merge',
                'name': 'merge',
                'package': 'fastcore',
                'path': '/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/fastcore/basics.py',
                'type': 'FunctionDef',
                'uploaded_at': 1776124140.6147513,
                'version': '1.12.39'},
  'node': 'fastcore.basics.merge',
  'out_degree': 12,
  'pagerank': 3e-05,
  'rank': -10.449557921383162,
  'rowid': 641}
```

In [ ]:
#| export
@patch
def dep_stack(self:Karma, seeds:list=None, depth:int=1) -> list:
	'BFS over pkg_deps (+ repo_deps if code_db given). Ordered by coupling strength.'
	seen, frontier, layers = set(seeds), set(seeds), [list(seeds)]
	for _ in range(depth):
		if not frontier: break
		pl = ','.join('?'*len(frontier))
		rows = self.env_pd(select='to_pkg tgt,n_modules n', where=f'from_pkg IN ({pl})', where_args=list(frontier))
		rows += self.code_rd(select='to_pkg tgt,n_files n', where=f'from_module IN ({pl})', where_args=list(frontier))
		nxt = {r['tgt'] for r in rows if r['tgt'] not in seen}
		if not nxt: break
		layers.append(sorted(nxt, key=lambda p: sum(r['n'] for r in rows if r['tgt']==p), reverse=True))
		seen |= nxt
		frontier = nxt
	return layers

@patch
def task_context(self:Karma, q:str, depth:int=3, limit:int=10) -> dict:
	'Structured context: candidate packages → dep stack → graph entry points → call paths.'
	pkgs = L(self.pkg_context(q, limit=limit)).map(lambda r: r['metadata']['name']).filter(lambda p: bool(spec(p)))
	layers = self.dep_stack(list(pkgs), depth=depth) if pkgs else []
	return dict(packages=pkgs, dep_layers=layers)

In [ ]:
# dep_stack smoke test
k2 = Karma(xdg_dir=Path('.'))
k2.update_pkg('fastlite')
rows = list(k2.envdb.t.pkg_deps(where='from_pkg="fastlite"'))
assert rows, 'pkg_deps should have rows after update_pkg'
assert any(r['to_pkg'] in ('fastcore','fastlite') for r in rows), f'expected fastcore/fastlite: {rows[:3]}'
print('pkg_deps via update_pkg ok:', rows[:3])

pkg_deps via update_pkg ok: [{'from_pkg': 'fastlite', 'to_pkg': 'apsw', 'n_modules': 1}, {'from_pkg': 'fastlite', 'to_pkg': 'apswutils', 'n_modules': 2}, {'from_pkg': 'fastlite', 'to_pkg': 'dataclasses', 'n_modules': 2}]


In [ ]:
layers = k2.dep_stack(['fastlite'])
print(layers)
print('\n\n-------------------------------------------------\n\n')
stack = L(layers).flatten()
assert 'fastcore' in stack, f'fastcore should be in dep_stack: {stack}'
print('_dep_stack ok:', stack)

[['fastlite'], ['apswutils', 'dataclasses', 'enum', 'typing', 'fastcore', 'inspect', 'types', 'apsw']]


-------------------------------------------------


_dep_stack ok: ['fastlite', 'apswutils', 'dataclasses', 'enum', 'typing', 'fastcore', 'inspect', 'types', 'apsw']


In [ ]:
tc = k2.task_context('payments page monsterui fasthtml')
print('task_context packages:', tc['packages'])
print('task_context dep_stack:', tc['dep_layers'])

task_context packages: ['fastcore', 'httpx', 'fastlite', 'apsw', 'ghapi', 'apswutils', 'litesearch']
task_context dep_stack: [['fastcore', 'httpx', 'fastlite', 'apsw', 'ghapi', 'apswutils', 'litesearch'], ['typing', 'os', 're', 'inspect', 'sys', 'functools', 'json', 'time', 'contextlib', 'collections', 'types', 'io', 'dataclasses', 'enum', 'urllib', 'importlib', 'warnings', 'random', 'argparse', 'ast', 'textwrap', 'math', 'copy', 'itertools', 'datetime', 'tempfile', 'pathlib', 'shutil', 'base64', 'contextvars', 'ssl', 'numpy', 'subprocess', 'html', 'asyncio', 'pprint', 'logging', 'codecs', 'fnmatch', 'csv', 'PIL', 'threading', 'shlex', 'traceback', 'mimetypes', 'http', 'pickle', 'hashlib', 'pdf_oxide', 'ipaddress', 'webbrowser', 'socket', 'xml', 'gzip', 'usearch', 'idna', 'atexit', 'difflib', 'uuid', 'string', 'struct', 'httpcore', 'fractions', 'certifi', 'resource', 'builtins', 'decimal', 'getpass', 'weakref', 'tokenize', 'pygments', 'readline', 'tomllib', 'queue', 'abc', 'tokenizers'

In [ ]:
k2.context('payments page monsterui fasthtml', repo=False)[:10]

[{'rowid': 1068, 'content': 'def _repr_html_(self:FT): return self.__html__()', 'metadata': {'path': '/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/fastcore/xml.py', 'uploaded_at': 1776124140.6237516, 'name': '_repr_html_', 'lang': '.py', 'type': 'FunctionDef', 'lineno': 224, 'end_lineno': 224, 'package': 'fastcore', 'version': '1.12.39', 'mod_name': 'fastcore.xml._repr_html_'}, '_dist': 0.7265547513961792, '_rrf_score': 0.016666666666666666, '_src_id': 'env:1068', 'node': 'fastcore.xml._repr_html_', 'flavor': 'function', 'file': '/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/fastcore/xml.py', 'pagerank': 0.00013, 'in_degree': 1, 'out_degree': 4, 'callers': ['fastcore.xml.FT'], 'callees': ['fastcore.xml.__html__', 'fastcore.basics.patch', 'fastcore.xml.FT', 'fastcore.xml.Safe.__html__'], 'co_dispatched': []}, {'rowid': 1067, 'content': 'def __html__(self:FT): return to_xml(self, indent=False)', 'metadata': {'path': '/Users/71293/cod

In [ ]:
k2.context('payments page packages:monsterui,python-fasthtml', repo=False)[:10]

[{'rowid': 1073, 'content': 'def showtags(s):\n    return f"""<code><pre>\n{escape(to_xml(s))}\n</code></pre>"""', 'metadata': {'path': '/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/fastcore/xml.py', 'uploaded_at': 1776124140.6237516, 'name': 'showtags', 'lang': '.py', 'type': 'FunctionDef', 'lineno': 248, 'end_lineno': 251, 'package': 'fastcore', 'version': '1.12.39', 'mod_name': 'fastcore.xml.showtags'}, '_dist': 0.7440280914306641, '_rrf_score': 0.016666666666666666, '_src_id': 'env:1073', 'node': 'fastcore.xml.showtags', 'flavor': 'function', 'file': '/Users/71293/code/personal/orgs/karma/.venv/lib/python3.13/site-packages/fastcore/xml.py', 'pagerank': 3e-05, 'in_degree': 0, 'out_degree': 2, 'callers': [], 'callees': ['None.escape', 'fastcore.xml.to_xml'], 'co_dispatched': []}, {'rowid': 205, 'content': 'def enable_pages(self:GhApi, branch=None, path="/"):\n    "Enable or update pages for a repo to point to a `branch` and `path`."\n    if path not in (\'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()